# LLM Generation Parameters

## Import Libraries

In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [4]:
# Always remember to do this! - What does this do?
load_dotenv(override=True)

True

## Exploring Parameters

### OpenAI Chat Completions API (most widely used)

`OpenAI` Official documentation is here - https://platform.openai.com/docs/api-reference/chat/create Refer to this for a full set of parameters that we'll later explore. 

The most used parameters are as listed below, with definitions from official `OpenAI` API documentation. These are considered "common" because they fundamentally control how an autoregressive language model samples tokens to generate output. 

- **max_tokens** (only *max_completion_tokens* is supported by o1 series; no default): The maximum number of completion tokens that may be used over the course of the run. The run will make a best effort to use only the number of completion tokens specified, across multiple turns of the run. If the run exceeds the number of completion tokens specified, the run will end with status `incomplete`.
- **temperature** (*1 by default*): What sampling temperature to use, between 0 and 2. Higher values like 0.8 will make the output more random, while lower values like 0.2 will make it more focused and deterministic. We generally recommend altering this or top_p but not both.
- **top_p** (*1 by default*): An alternative to sampling with temperature, called nucleus sampling, where the model considers the results of the tokens with top_p probability mass. So 0.1 means only the tokens comprising the top 10% probability mass are considered.
We generally recommend altering this or temperature but not both.
- **top_k**: not available in OpenAI client library
- **frequency_penalty** (*0 by default*): Number between -2.0 and 2.0. Positive values penalize new tokens based on their existing frequency in the text so far, decreasing the model's likelihood to repeat the same line verbatim.
- **presence_penalty** (*0 by default*): Number between -2.0 and 2.0. Positive values penalize new tokens based on whether they appear in the text so far, increasing the model's likelihood to talk about new topics.
- **stop** (*null by default*): Not supported with latest reasoning models o3 and o4-mini. Up to 4 sequences where the API will stop generating further tokens. The returned text will not contain the stop sequence.

### Anthropic's Messages API

It's good to note that Anthropic's `Messages` API contains similar parameters except a few differences:

- **max_tokens** is used as is
- **temperature** has a value between 0 and 1
- **top_p** is as is
- **top_k** is also available: Only sample from the top K options for each subsequent token. Used to remove "long tail" low probability responses. Recommended for advanced use cases only. You usually only need to use temperature.
- **frequency_penalty** and **presence_penalty** parameters are not available.
- **stop_sequences** is used instead of **stop**

### Some important considerations

- Exact naming conventions of parameters can vary, but functionality is same (like we saw with Anthropic's API).
- The optimal range or impact of a parameter might differ between models, even if general principle is same. A temperature of 1.0 might be very wild for one model but only moderately creative for another.
- Differences between different parameter configurations will be more prominent in larger LLMs, as with smaller LLMs, the "spread" of probabilities for the next token might not be as wide.

### Setting temperature vs. top_p for different use cases: 

| Use Case               | Temperature | Top_p | Description                                                                                        |
| :--------------------- | :---------- | :---- | :------------------------------------------------------------------------------------------------- |
| Code Generation        | 0.2         | 0.1   | Generates code that adheres to established patterns and conventions. Output is more deterministic and focused. Useful for generating syntactically correct code. |
| Creative Writing       | 0.7         | 0.8   | Generates creative and diverse text for storytelling. Output is more exploratory and less constrained by patterns. |
| Chatbot Responses      | 0.5         | 0.5   | Generates conversational responses that balance coherence and diversity. Output is more natural and engaging. |
| Code Comment Generation | 0.3         | 0.2   | Generates code comments that are more likely to be concise and relevant. Output is more deterministic and adheres to conventions. |
| Data Analysis Scripting | 0.2         | 0.1   | Generates data analysis scripts that are more likely to be correct and efficient. Output is more deterministic and focused. |
| Exploratory Code Writing | 0.6         | 0.7   | Generates code that explores alternative solutions and creative approaches. Output is less constrained by established patterns. |

References from here: https://community.openai.com/t/cheat-sheet-mastering-temperature-and-top-p-in-chatgpt-api/172683/10

### Setting Up

In [3]:
# Use either Llama3.2 (3B) with Ollama
!ollama pull llama3.2    

# Initialize the OpenAI client to connect to Ollama
llm_api = OpenAI(base_url='http://localhost:11434/v1', 
                 api_key='ollama') # note that llm_api is a good variable name, because the LLM is indeed exposed locally as an API by Ollama!

# Define the model to use
model_name = "llama3.2" # Ensure this matches the pulled model in Ollama

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
# # Or use DeepSeek-R1-Distill-Llama-70B from Groq
# llm_api = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# # Define the model to use
# model_name = "deepseek-r1-distill-llama-70b"

In [5]:
#Helper function for text generation
def generate_and_display(prompt, param_name, param_value, num_runs=1, **kwargs):
    """
    Helper function to make a call to the LLM and display results.
    Runs multiple times for stochastic parameters to highlight differences.
    """
    print(f"\n--- Demonstrating {param_name}={param_value} ---")
    print(f"Prompt: \"{prompt}\"")
    print(f"Other consistent parameters: {kwargs}")

    messages = [
        {"role": "system", "content": "You are a helpful and concise AI assistant for a business."},
        {"role": "user", "content": prompt},
    ]

    for i in range(num_runs):
        print(f"\n--- Run {i+1} ---")
        try:
            response = llm_api.chat.completions.create(
                model=model_name,
                messages=messages,
                **{param_name: param_value, **kwargs}
            )
            print("Generated Text:")
            answer = response.choices[0].message.content.strip()
            display(Markdown(answer)) # .strip() removes leading/trailing whitespace for cleaner output
        except Exception as e:
            print(f"An error occurred: {e}")
            print("Ensure Ollama is running and the model is pulled. Skipping further runs for this parameter setting.")
            break # Exit loop if an error occurs

    print("\n--- End Demonstration for this parameter setting ---\n" + "="*80)

### 1. Temperature

**Commercial Use Cases**: Content generation (marketing copy, ad variations, blog post ideas), creative brainstorming for product features, unique customer engagement messages.

**What to expect:**
- **Low Temperature**: Look for very similar, factual, and direct outputs across runs. Less "fluff," more to the point.
    - E.g., Consistent, factual product bullet points for a catalog
- **Medium Temperature**: Outputs will start showing more natural language, some variation in phrasing, but still coherent.
    - E.g., Engaging marketing copy for a landing page
- **High Temperature**: Expect more varied vocabulary, potentially more metaphorical or unusual phrasing, and noticeable differences between runs. It might sometimes deviate slightly from the core prompt if pushed too high.
    - E.g., Brainstorming unique ad slogans or creative concepts for a campaign

In [6]:
common_prompt_temp = """
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."""

In [7]:
# Example 1.1: Low Temperature
generate_and_display(
    common_prompt_temp,
    "temperature", 0.1,
    num_runs=3, # Run multiple times to show consistency
    max_completion_tokens=70
)


--- Demonstrating temperature=0.1 ---
Prompt: "
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."
Other consistent parameters: {'max_completion_tokens': 70}

--- Run 1 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that empowers patients to take control of their health. Key features include:

* **Personalized Health Profiles**: AI-driven analysis of patient data, medical history, and lifestyle habits creates a unique profile.
* **Predictive Insights**: AI forecasts potential health risks and provides actionable recommendations for prevention and treatment.
* **Virtual Health Coaching**: AI-powered chatbots offer guidance on healthy habits, nutrition, and wellness.

**AI Gateway Enabling Production-Ready Solution**

The AI gateway plays a crucial role in integrating patient data from various sources, ensuring seamless communication between Merck Vitals and external systems. This enables:

* **Data Standardization**: Normalizing patient data for accurate analysis and insights.
* **Integration with Wearables & Devices**: Connecting patients' wearable devices and health trackers to provide comprehensive data.
* **Secure Data Storage**: Safeguarding sensitive patient information in compliance with regulatory standards.

By leveraging the AI gateway, Merck Vitals can deliver a robust, production-ready solution that revolutionizes personalized healthcare for millions of patients.


--- Run 2 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that empowers patients to take control of their health. Key features include:

* **Personalized Health Profiles**: AI-driven analysis of patient data, medical history, and lifestyle habits.
* **Predictive Insights**: Proactive alerts for potential health risks and preventive measures.
* **Medication Optimization**: AI-suggested medication regimens based on individual needs.

**AI Gateway Enabling Production-Ready Solution**

The AI gateway plays a crucial role in integrating Merck Vitals with various healthcare systems, ensuring seamless data exchange and secure patient information management. The gateway enables:

* **Data Standardization**: Normalizing patient data for analysis and processing.
* **Integration with Wearables & Devices**: Connecting patients' wearable devices to provide real-time health insights.
* **Secure Data Storage**: Safeguarding sensitive patient data in compliance with regulatory standards.

By leveraging the AI gateway, Merck Vitals becomes a production-ready solution, providing a comprehensive and personalized healthcare experience for millions of patients.


--- Run 3 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that empowers patients to take control of their health. Key features include:

* **Personalized Health Profiles**: AI-driven analysis of patient data, medical history, and lifestyle habits creates a unique profile.
* **Predictive Insights**: AI forecasts potential health risks and provides actionable recommendations for prevention and treatment.
* **Virtual Health Coaching**: AI-powered chatbots offer guidance on healthy habits, nutrition, and wellness.

**AI Gateway Enabling Production-Ready Solution**

The AI gateway plays a crucial role in integrating patient data from various sources, ensuring seamless communication between Merck's systems and external healthcare providers. This enables:

* **Data Standardization**: Normalization of patient data for accurate analysis.
* **Secure Data Exchange**: Secure transmission of sensitive patient information.
* **Scalability**: Scalable architecture to accommodate growing user base.

By leveraging the AI gateway, Merck Vitals becomes a production-ready solution, providing patients with personalized healthcare insights and empowering them to make informed decisions about their well-being.


--- End Demonstration for this parameter setting ---


In [8]:
# Example 1.2: Medium Temperature
generate_and_display(
    common_prompt_temp,
    "temperature", 0.7,
    num_runs=3, # Run multiple times to show controlled variation
    max_completion_tokens=70
)


--- Demonstrating temperature=0.7 ---
Prompt: "
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."
Other consistent parameters: {'max_completion_tokens': 70}

--- Run 1 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that combines patient data with medical knowledge to provide personalized health insights. Key features include:

* AI-driven health risk assessments
* Tailored treatment recommendations based on individual needs
* Real-time monitoring and alerts for early interventions

This innovative service will enhance daily life by empowering users to take control of their health, making informed decisions about their well-being.

**AI Gateway Role**

The AI gateway plays a crucial role in enabling a production-ready solution. It acts as an intermediary between Merck's Agentic AI technology and the healthcare system, ensuring seamless data integration, security, and scalability. The gateway will facilitate secure data exchange, standardize data formats, and enable real-time updates to improve treatment efficacy.


--- Run 2 ---
Generated Text:


**Merck Vitals:**
Innovative Features:

* AI-driven health assessments and personalized recommendations
* Integration with wearable devices and electronic health records
* Advanced disease prediction and prevention tools
* Virtual consultations with Merck experts and medical professionals

Enhancing daily life for a typical user, "Merck Vitals" streamlines healthcare by providing actionable insights, reducing doctor visits, and promoting proactive self-care. Users can monitor their health, receive tailored advice, and access expert support.

**AI Gateway:**
Enabling production-ready solution:
The AI gateway acts as the conduit between disparate data sources, integrating patient data from various platforms and medical knowledge repositories. This ensures seamless communication with Merck's Agentic AI algorithms, facilitating the development of accurate models for personalized healthcare predictions and recommendations.


--- Run 3 ---
Generated Text:


Merck Vitals is an Agentic AI-powered service that empowers personalized healthcare through data-driven insights.

Innovative features:

* Proactive health monitoring: AI analyzes patient data to detect anomalies and alert users to potential health risks
* Tailored treatment plans: AI-generated recommendations based on individual patient needs and medical knowledge
* Virtual health coaching: AI-powered guidance for healthy lifestyle choices and disease management

AI Gateway plays a crucial role in enabling production-ready solutions by:

* Integrating disparate patient data sources
* Validating data quality and accuracy
* Providing a unified interface for healthcare professionals to access and act on patient insights


--- End Demonstration for this parameter setting ---


In [9]:
# Example 1.3: High Temperature
generate_and_display(
    common_prompt_temp,
    "temperature", 1.5, # Pushing temperature higher for more dramatic effect
    num_runs=3, # Run multiple times to show increased randomness
    max_completion_tokens=70
)


--- Demonstrating temperature=1.5 ---
Prompt: "
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."
Other consistent parameters: {'max_completion_tokens': 70}

--- Run 1 ---
Generated Text:


**Merck Vitals**

Envisioned as a self-management AI-powered platform, Merck Vitals empowers patients to prioritize individualized health through actionable insights and personalized guidance. Innovative features include:

* Personalized risk assessment profiles tailored to patients' medical histories
* Context-driven symptom identification and diagnosis support via conversational interfaces or wearable devices
* AI-generated recommendations for lifestyle modifications and medications as needed

**Integrated AI Gateways**

To deliver this innovation, a sophisticated AI gateway system would be instrumental. The gateway will seamlessly manage:

- Seamless integration of clinical data streams (EDM records, lab results) with patient-generated content using AI gateways bridge internal siloed systems
AI-powered real-time updates on diagnoses allowing health provider coordination


--- Run 2 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals harnesses patient data, medical knowledge, and Agentic AI to empower individualized healthcare.

- Personalized health insights via immersive reports
- Real-time disease risk prediction and alerting
- Intelligent symptom checker for tailored guidance
- Predictive care pathways for optimal wellness routines

AI Gateway accelerates Merck Vitals by securing patient data and integrating emerging technologies, ensuring seamless deployment of innovative Agentic AI capabilities.


--- Run 3 ---
Generated Text:


**Merck Vitals: Personalized Healthcare**

Merck Vitals is an Agentic AI-powered service that harnesses patient data and medical knowledge to deliver personalized healthcare solutions.

* **Innovative Features:** Emantic Interventions (EAIs) identify individualized prevention, early detection, and treatment actions.
* **Improved Outcomes:** AI-optimized health guidance supports daily living, streamlining care transitions.
*Enhanced Health: Customized wellness plans combine genomic, digital, lab data insights for effective preventive care.



To enable production-readiness, a role of the Merck Agnetic gateway (AI gateways) to handle scalability across healthcare settings worldwide is vital enabling multi-decade longevity life and improved user experiences.


--- End Demonstration for this parameter setting ---


### 2. Max_tokens

**Commercial Use Cases**: Chatbot response length control, email subject line generation, tweet generation, summarizing lengthy documents for internal reports, generating specific-length ad copy.

**What to expect**: 

- The generated output will strictly adhere to the `max_tokens` limit, regardless of whether the summary is complete.
- Depending on the use case, the `max_tokens` parameter can be set:
    - Very few tokens (e.g., for a headline or quick alert)
    - Moderate tokens (e.g., for an executive summary or internal Slack message)
    - More tokens (e.g., for a detailed summary section in a newsletter or report)

In [10]:
common_prompt_max_tokens = """
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."""

In [11]:
# Example 2.1: Very few tokens
generate_and_display(
    common_prompt_max_tokens + " Complete response in less than 15 tokens.",
    "max_tokens", 15,
    temperature=0.5
)


--- Demonstrating max_tokens=15 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles. Complete response in less than 15 tokens."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist, I'd leverage Generative AI to:

1.


--- End Demonstration for this parameter setting ---


In [12]:
# Example 2.2: Moderate tokens
generate_and_display(
    common_prompt_max_tokens,
    "max_tokens", 50,
    temperature=0.5
)


--- Demonstrating max_tokens=50 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist working with an IPL team, I'd love to explore the potential of Generative AI in creating innovative training drills and strategic simulations for batsmen. Here's how:

**Novel Training Drills:**

1. **Pitch-specific


--- End Demonstration for this parameter setting ---


In [13]:
# Example 2.3: More tokens
generate_and_display(
    common_prompt_max_tokens,
    "max_tokens", 150,
    temperature=0.5
)


--- Demonstrating max_tokens=150 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist working with an IPL team, I'd love to explore the potential of Generative AI (GA) in creating novel training drills and strategic simulations for batsmen. Here's how GA could be utilized:

**Training Drills:**

1. **Adaptive Batting Simulation**: Utilize GA algorithms like Generative Adversarial Networks (GANs) or Variational Autoencoders (VAEs) to generate realistic batting scenarios based on historical data, pitch conditions, and bowling styles. These simulations can mimic the challenges faced by batsmen on Indian pitches.
2. **Drill Generation**: Leverage GA to create personalized training drills for individual batsmen. By analyzing their strengths, weaknesses, and playing style, GA can design customized


--- End Demonstration for this parameter setting ---


In [14]:
#Example 2.4: Even more max_tokens
generate_and_display(
    common_prompt_max_tokens,
    "max_tokens", 1000,
    temperature=0.5
)


--- Demonstrating max_tokens=1000 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist working with an IPL team, I'd like to explore the potential of Generative AI in creating novel training drills and strategic simulations for batsmen. Here's a comprehensive approach to harnessing this technology:

**Data Collection**

To develop effective Generative AI models, we'll need access to large datasets containing various bat-ball interactions, including:

1. **IPL match data**: Collect historical data from IPL matches, including ball-by-ball information, batting performances, and bowling strategies.
2. **Batting technique analysis**: Analyze expert assessments of batsmen's techniques, such as footwork, strokeplay, and shot selection.
3. **Bowling style classification**: Classify different bowling styles (e.g., fast bowler, spin bowler, swing bowler) based on their action, pace, and trajectory.

**Generative AI Models**

We'll employ Generative Adversarial Networks (GANs) or Variational Autoencoders (VAEs) to create novel training drills and strategic simulations. These models will learn from our dataset and generate new scenarios that mimic real-game conditions.

1. **Batting simulation**: Use GANs or VAEs to simulate different batting situations, such as:
	* Scenarios with unique Indian pitches (e.g., Doosra, Carrom ball).
	* Different bowling styles and their variations.
	* Various weather conditions (e.g., rain, humidity).
2. **Drill generation**: Employ a Generative Model to create novel training drills based on the simulated scenarios:
	* Batting drills with varying difficulty levels.
	* Drills focusing on specific skills, such as shot selection or footwork.

**Strategic Simulation**

To develop strategic simulations, we'll use VAEs or Recurrent Neural Networks (RNNs) to analyze historical data and predict optimal batting strategies:

1. **Batting strategy prediction**: Use VAEs or RNNs to predict the best batting strategy for a given situation:
	* Batting style (e.g., defensive, aggressive).
	* Shot selection.
	* Field placement.
2. **Bowling strategy optimization**: Employ an RNN-based model to optimize bowling strategies based on historical data and real-time game conditions.

**Implementation**

To integrate these models into our training program:

1. **Data visualization tools**: Utilize data visualization tools (e.g., Tableau, Power BI) to present the simulated scenarios and strategic predictions in a user-friendly format.
2. **Training equipment**: Develop custom training equipment that can simulate the conditions generated by the Generative AI models.
3. **Coaching integration**: Collaborate with coaches to integrate these models into their training programs, ensuring that batsmen are trained on realistic scenarios and strategies.

**Benefits**

The use of Generative AI in creating novel training drills and strategic simulations for batsmen will:

1. **Enhance batting skills**: Provide batsmen with a more comprehensive understanding of different bowling styles and conditions.
2. **Improve decision-making**: Offer coaches and batsmen data-driven insights to optimize batting strategies.
3. **Increase adaptability**: Help batsmen develop the ability to adapt quickly to changing game conditions.

By leveraging Generative AI, we can create a cutting-edge training program that empowers our batsmen to perform at their best in the IPL and beyond.


--- End Demonstration for this parameter setting ---


### 3. Top_p (Nucleus sampling)

**Commercial Use Cases**: Generating diverse customer service responses (without going off-topic), creating varied product names, generating slightly different content variations for A/B testing.

**What to Observe**:

- **Low** top_p: Outputs will be very similar across runs, often picking the most statistically common words/phrases. Less variation in phrasing.

- **Medium** top_p: Some variations in phrasing will emerge, but still within a coherent and expected range.

- **High** top_p: More diverse and potentially more "creative" word choices and phrasing, showing greater variation across runs.

In [15]:
common_prompt_top_p = """
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."""

In [16]:
# Example 3.1: Low top_p (for a very standard, consistent customer service greeting)
generate_and_display(
    common_prompt_top_p,
    "top_p", 0.1,
    num_runs=3,
    temperature=0.7,
    max_tokens=80
)


--- Demonstrating top_p=0.1 ---
Prompt: "
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 80}

--- Run 1 ---
Generated Text:


"Welcome to [Your Business Name]! We're thrilled you're interested in trying our new premium coffee blend. I'd be happy to tell you more about it.

Our expertly crafted blend is indeed a game-changer, and we're confident it will rival the best of the big names like Starbucks, Peets Coffee, and Cafe Coffee Day. What sets us apart, however, is its


--- Run 2 ---
Generated Text:


"Welcome to [Your Business Name]! We're thrilled you're interested in trying our new premium coffee blend. I'd be happy to tell you more about it.

Our expertly crafted blend is indeed a game-changer, and we're confident it will rival the best of the big names like Starbucks, Peets Coffee, and Cafe Coffee Day. What sets us apart, however, is its


--- Run 3 ---
Generated Text:


"Welcome to [Your Business Name]! We're thrilled you're interested in trying our new premium coffee blend. I'd be happy to tell you more about it.

Our expertly crafted blend is indeed a game-changer, and we're confident it will rival the best of the big names like Starbucks, Peets Coffee, and Cafe Coffee Day. What sets us apart, however, is its


--- End Demonstration for this parameter setting ---


In [17]:
# Example 3.2: Medium top_p (for a friendly, slightly varied customer service interaction)
generate_and_display(
    common_prompt_top_p,
    "top_p", 0.5,
    num_runs=3,
    temperature=0.7,
    max_tokens=80
)


--- Demonstrating top_p=0.5 ---
Prompt: "
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 80}

--- Run 1 ---
Generated Text:


Here's a warm and inviting response to the customer's inquiry:

"Thank you for considering our premium coffee blend! We're thrilled to introduce our newest offering, which has been carefully crafted to rival the best of the industry - Starbucks, Peets Coffee, and Cafe Coffee Day.

Our expert roasters have worked tirelessly to create a unique flavor profile that will tantalize your taste buds. Our signature blend


--- Run 2 ---
Generated Text:


Here's a warm and inviting response to the customer's inquiry:

"Thank you for your interest in our new premium coffee blend! We're thrilled to introduce our latest creation, which has been carefully crafted to rival the best of the industry. Our expert roasters have carefully selected a unique blend of beans from around the world, resulting in a rich and complex flavor profile that's sure to tantalize your


--- Run 3 ---
Generated Text:


"Welcome to [Your Business Name]! We're thrilled you're interested in our new premium coffee blend. I'd be happy to tell you more about it.

Our signature blend is carefully crafted to rival the best of the industry, and we're confident you'll love its rich, smooth flavor profile. With notes of dark chocolate, hints of caramel, and a subtle hint of smokiness,


--- End Demonstration for this parameter setting ---


In [18]:
# Example 3.3: High top_p (for brainstorming diverse opening lines for marketing outreach)
generate_and_display(
    common_prompt_top_p,
    "top_p", 0.99, # Pushing higher for more noticeable effect
    num_runs=3,
    temperature=0.7,
    max_tokens=80
)


--- Demonstrating top_p=0.99 ---
Prompt: "
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 80}

--- Run 1 ---
Generated Text:


Here's a potential response:

"Thank you so much for considering our premium coffee blend! We're thrilled to introduce you to our newest creation, designed to rival the best of the industry. Our expertly crafted blend is carefully curated from the finest Arabica beans, sourced from around the world to bring out the perfect balance of flavors.

What sets our blend apart is its unique flavor profile, which combines


--- Run 2 ---
Generated Text:


Here's a warm and inviting response to welcome the customer and highlight your premium coffee blend:

"Hello! I'm thrilled you're interested in our new premium coffee blend! We've worked tirelessly with our expert roasters to craft a truly exceptional blend that's sure to satisfy even the most discerning coffee lovers.

Our unique 'Tuscan Sunrise' blend is inspired by the rich flavors of Italy and


--- Run 3 ---
Generated Text:


"Welcome to our store! We're thrilled you're interested in our new premium coffee blend. I'd be happy to tell you more about it.

Our signature blend is crafted with the finest, small-batch Arabica beans from around the world, carefully selected for their distinct flavor profiles. What sets our blend apart is its unique combination of notes - rich chocolate hints, subtle fruity undertones, and


--- End Demonstration for this parameter setting ---


### 4. Frequency_penalty

**Commercial Use Cases**: Ensuring variety in marketing emails, preventing chatbots from repeating FAQs, generating diverse social media posts about the same product.

**What to Observe**:

- **No** penalty (0.0): Look for words or short phrases being repeated frequently within the generated paragraph.

- **Moderate** penalty: Repetition should be noticeably reduced, leading to more varied sentences.

- **High** penalty: The model will actively avoid repeating words, potentially leading to more complex or less natural phrasing if it has to find many synonyms.

In [19]:
common_prompt_freq_penalty = """
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"""

In [20]:
# Example 4.1: Negative frequency penalty (encourages repetition)
generate_and_display(
    common_prompt_freq_penalty,
    "frequency_penalty", -1,
    temperature=0.7,
    seed = 42
)


--- Demonstrating frequency_penalty=-1 ---
Prompt: "
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"
Other consistent parameters: {'temperature': 0.7, 'seed': 42}

--- Run 1 ---
Generated Text:


1. "I'd be happy to assist you further."
2. "Can you please provide more details?"
3. "I apologize for the inconvenience."
4. "Let me check on that for you."
5. "Our policy is as follows..."
6. "I understand your concern, and I'm here to help."
7. "I'll do my best to resolve this issue."
8. "Can you please confirm your order number?"
9. "I'll escalate this to my supervisor."
10. "I apologize for the delay."
11. "I'd like to offer a solution."
12. "I'll need to verify some information."
13. "I'll provide you with an update."
14. "Is there anything else I can assist you with?"
15. "I'll need to authenticate your account."


--- End Demonstration for this parameter setting ---


In [21]:
# Example 4.2: Moderate frequency penalty (reduced repetition)
generate_and_display(
    common_prompt_freq_penalty,
    "frequency_penalty", 1.0, # Increased for more impact
    temperature=0.7,
    seed=42
)


--- Demonstrating frequency_penalty=1.0 ---
Prompt: "
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"
Other consistent parameters: {'temperature': 0.7, 'seed': 42}

--- Run 1 ---
Generated Text:


1. "I'd be happy to assist you further."
2. "Can you please provide more details?"
3. "Let me see what I can do."
4. "Our policy is..."
5. "I apologize for the inconvenience."
6. "Would you like me to escalate this issue?"
7. "That's not possible at this time."
8. "May I ask a few questions?"
9. "Is there anything else I can help with today?"
10. "Can you confirm your order number please?"
11. "I'll check on that for you immediately."
12. "Our current status is..."
13. "What was the issue again?"
14. "Would you prefer to resolve this now or at a later time?"
15. "Is there anything else I can do right now?"


--- End Demonstration for this parameter setting ---


In [22]:
# Example 4.3: High frequency penalty (strong reduction in repetition, might affect coherence)
generate_and_display(
    common_prompt_freq_penalty,
    "frequency_penalty", 2.0, # Max penalty for strong effect
    temperature=0.7,
    seed=42
)


--- Demonstrating frequency_penalty=2.0 ---
Prompt: "
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"
Other consistent parameters: {'temperature': 0.7, 'seed': 42}

--- Run 1 ---
Generated Text:


1. "I'd be happy to assist you further."
2. "Can I provide an alternative solution?"
3. "Let me look into this for you."
4."Our policy is as follows..."
5."Would you like me to escalate this issue?"
6.'"We apologize, but [specific reason]."
7."'s the status of your order?
8.""How did we do today? Is there anything else I can help with?"
9. "I'm going to need some more information from you."
10. "Our team will get back to you within 24 hours."
11."Is this an issue or a question?"
12."'s the next step in resolving your concern?
13.""We're here for our valued customers..."
14.""How can I make it right for you today?"
15. ""Let me check on that availability"


--- End Demonstration for this parameter setting ---


### 5. Presence_penalty

**Commercial Use Cases**: Encouraging a customer service bot to explore different solutions/topics, generating comprehensive meeting minutes by covering all points, producing diverse content ideas for a campaign.

**What to Observe**:

- **No** penalty (0.0): The model might dwell on a few initial topics or points without moving on to others mentioned in the conceptual prompt.

- **Moderate** penalty: The summary should cover a broader range of distinct action items or concepts, rather than just elaborating on the first few.

- **High** penalty: The model will try hard to introduce new concepts or distinct ideas, possibly jumping between points quickly or even generating somewhat disjointed output if forced too much.

In [23]:
common_prompt_pres_penalty = """
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"""

In [24]:
# Example 5.1: No presence penalty (might focus heavily on one or two main themes)
generate_and_display(
    common_prompt_pres_penalty,
    "presence_penalty", 0.0,
    temperature=0.7,
)


--- Demonstrating presence_penalty=0.0 ---
Prompt: "
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"
Other consistent parameters: {'temperature': 0.7}

--- Run 1 ---
Generated Text:


Here's a brief overview of four distinct areas where Generative AI (GenAI) can significantly improve our sales and customer engagement processes, along with examples:

**1. Personalized Sales Content Generation**

By leveraging GenAI, we can generate personalized sales content that resonates with individual customers. For instance:

* We'll develop an AI-powered tool that analyzes a customer's purchase history, interests, and behavior to create customized sales emails or social media posts.
* Output: "Hey [Customer], we noticed you've been reading about [related topic]. Our new product is just what you need..."

**2. Chatbot-Driven Customer Conversations**

GenAI-powered chatbots can engage with customers in a more human-like way, providing instant support and answers to common questions. Example:

* We'll create an AI-driven chatbot that uses natural language processing (NLP) to understand customer inquiries.
* Output: "Hi! Can I help you find the best deal on [product]? Our current offers are limited to [amount]."

**3. Predictive Lead Scoring and Qualification**

GenAI can analyze vast amounts of data to predict lead behavior, helping us qualify and prioritize leads more efficiently. For example:

* We'll develop an AI model that analyzes a prospect's online activity, social media engagement, and purchase history to score their likelihood of converting.
* Output: "Based on our analysis, we're confident that [Lead] has a 90% chance of converting within the next 30 days."

**4. Content Generation for Social Media and Blogs**

GenAI can help us generate high-quality content at scale, such as social media posts, blog articles, or product descriptions. Example:

* We'll use GenAI to create engaging social media posts that highlight a customer's success story.
* Output: "Meet [Customer], who just achieved [ achievement] with our innovative solution! Read their inspiring story here: [link]

These examples demonstrate how GenAI can enhance various aspects of sales and customer engagement, from personalized content generation to predictive lead scoring and more.


--- End Demonstration for this parameter setting ---


In [25]:
# Example 5.2: Moderate presence penalty (encourages covering a broader range of distinct action items)
generate_and_display(
    common_prompt_pres_penalty,
    "presence_penalty", 1.0, # Increased for more impact
    temperature=0.7,
)


--- Demonstrating presence_penalty=1.0 ---
Prompt: "
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"
Other consistent parameters: {'temperature': 0.7}

--- Run 1 ---
Generated Text:


Here's an overview of four key areas where Generative AI (GenAI) can make a significant impact on our sales and customer engagement processes:

**1. Personalized Content Generation**

* Example: Our marketing team can utilize GenAI to create personalized email campaigns, social media posts, or blog articles that cater to individual customers' interests and preferences.
	+ Benefits: Increased engagement rates, improved conversion rates, and enhanced brand relevance.

**2. Chatbot-Driven Customer Support**

* Example: We can leverage GenAI to develop advanced chatbots that use natural language processing (NLP) to provide instant support to our customers, helping them resolve common issues quickly and efficiently.
	+ Benefits: Reduced customer complaints, faster issue resolution times, and improved overall satisfaction.

**3. Predictive Lead Scoring**

* Example: By integrating GenAI with our sales data analytics platform, we can create predictive models that identify high-value leads based on their behavior, preferences, and past interactions with our brand.
	+ Benefits: Improved lead quality, enhanced sales productivity, and more accurate forecasting of future demand.

**4. Automated Sales Content Creation**

* Example: Our sales team can use GenAI to generate personalized product descriptions, data sheets, or presentation materials that highlight the unique value proposition of our products or services.
	+ Benefits: Increased pitch efficiency, reduced content creation time, and improved ability to demonstrate product benefits to potential customers.

These are just a few examples of how Generative AI can transform our sales and customer engagement processes. By implementing these strategies, we can unlock new opportunities for growth, improvement, and innovation in our business.


--- End Demonstration for this parameter setting ---


In [26]:
# Example 5.3: High presence penalty (strong encouragement for new ideas, potentially disjointed)
generate_and_display(
    common_prompt_pres_penalty,
    "presence_penalty", 2.0, # Max penalty for strong effect
    temperature=0.7,
)


--- Demonstrating presence_penalty=2.0 ---
Prompt: "
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"
Other consistent parameters: {'temperature': 0.7}

--- Run 1 ---
Generated Text:


Let's dive into the potential of Generative AI in improving our sales and customer engagement processes.

Here are four key areastrategies where GenAI can make a significant impact:

1. **Chatbot Personalization**

GenAI-powered chatbots can analyze customer interactions, preferences, and purchase history to provide tailored responses and recommendations.
Example: Using natural language processing (NLP), we created a chatbot that suggested products based on customers' browsing behavior, resulting in a 25% increase in average order value.

2. **Content Generation**

GenAI-powered content generators can help create high-quality sales materials, such as product descriptions, social media posts, and email campaigns.
Example: We used GenAI to generate engaging product videos that increased click-through rates by 30%. The AI algorithm analyzed customer feedback and sentiment data to identify key features and benefits.

3. **Product Recommendations**

GenAI-powered recommendation engines can analyze large datasets of customers' purchase history and preferences to provide personalized recommendations.
Example: By integrating a GenAI-driven recommendation engine into our e-commerce platform, we noticed a 15% increase in sales from recommended products that the customer had shown interest in before making a purchase.

4. **Sales Script Personalization**

GenAI-powered script generators can analyze sales data and create customized scripts for sales representatives to improve their performance.
Example: Using GenAI, we generated personalized sales scripts based on individual customers' pain points and buying behaviors. The result was an average increase of 20% in conversion rates during phone calls.

These strategies highlight the potential for Generative AI to enhance our internal processes and drive business growth through improved customer engagement and experience.


--- End Demonstration for this parameter setting ---


### 6. Stop

**Commercial Use Cases**: Ensuring chatbot responses don't continue past a natural break, extracting specific data blocks (e.g., JSON), generating bulleted lists that don't overextend, completing fill-in-the-blank forms.
garding:"

**What to Observe**: The model's generation will immediately halt once any of the provided stop sequences appear in its output.

In [27]:
# Example 6.1: Stopping at newline (for single-line completions)
common_prompt_stop = """
Draft a quick customer service response template:\nThank you for contacting TrueFoundry. We received your inquiry regarding:"""
generate_and_display(
    common_prompt_stop,
    "stop", ["\n"],
    temperature=0.5
)


--- Demonstrating stop=['\n'] ---
Prompt: "
Draft a quick customer service response template:
Thank you for contacting TrueFoundry. We received your inquiry regarding:"
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


Here's a draft of a quick customer service response template:


--- End Demonstration for this parameter setting ---


In [28]:
# Example 6.2: Stopping at a specific closing phrase (for structured email generation)
common_prompt_email = """Draft a professional email to a client regarding their contract status. 
Subject: Your Contract #12345 Update\nDear [Client Name],\n\nYour recent contract #12345 has been:"""
generate_and_display(
    common_prompt_email,
    "stop", ["Best regards,"], # Model will stop before adding the closing
    temperature=0.5
)


--- Demonstrating stop=['Best regards,'] ---
Prompt: "Draft a professional email to a client regarding their contract status. 
Subject: Your Contract #12345 Update
Dear [Client Name],

Your recent contract #12345 has been:"
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


Here's a draft of the email:

Subject: Your Contract #12345 Update

Dear [Client Name],

I am writing to provide you with an update on the status of your contract, #12345. As we previously discussed, our team has been working diligently to ensure that all terms and conditions are met.

As of today, I am pleased to inform you that your contract is currently: [insert current status, e.g. "awaiting final approval", "in review", "completed", etc.].

If you have any questions or concerns regarding the status of your contract, please do not hesitate to reach out to me directly. We will continue to work closely with you to ensure a smooth and successful partnership.

Please find attached a copy of the updated contract details for your reference.

Thank you for entrusting us with your business needs.


--- End Demonstration for this parameter setting ---


In [29]:
# Example 6.3: Stopping after a list item (for controlled list generation)
common_prompt_list = "List 3 benefits of our AI gateway:\n1. AI Governance\n2. Efficient budgeting\n3."
generate_and_display(
    common_prompt_list,
    "stop", ["4."], # Stop before generating the 4th item (assuming it would start with 4.)
    temperature=0.5
)


--- Demonstrating stop=['4.'] ---
Prompt: "List 3 benefits of our AI gateway:
1. AI Governance
2. Efficient budgeting
3."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


Here are three benefits of your AI gateway:

1. **AI Governance**: Our AI gateway provides a centralized platform to manage, monitor, and govern AI systems across the organization, ensuring that AI is used in a responsible and compliant manner.

2. **Efficient Budgeting**: The AI gateway offers automated budgeting and cost allocation capabilities, enabling businesses to easily track and optimize their AI-related expenses, making it easier to make informed financial decisions.

3. **Improved Data Security**: Our AI gateway features advanced data security measures, protecting sensitive business data from unauthorized access or breaches, ensuring the integrity and confidentiality of critical information assets.


--- End Demonstration for this parameter setting ---


### 7. Seed

**Commercial Use Cases**: Reproducible content for A/B testing (ensuring differences are due to prompt, not randomness), consistent chatbot behavior for quality assurance, debugging prompt engineering, generating consistent test data.

**What to Observe**:

- **Same** seed: The outputs for the multiple runs should be identical or extremely close.
- **Different** seed: The outputs, while still relevant to the prompt, should be distinct from each other across different seeds.

In [30]:
common_prompt_seed = "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."

In [31]:
# Example 7.1: Same seed, same prompt, same parameters (for reproducible slogan generation for testing)
print("\n--- Testing reproducibility with the SAME seed (42) ---")
generate_and_display(
    common_prompt_seed,
    "seed", 42,
    num_runs=3, # Run multiple times to show consistency
    temperature=0.7,
    max_tokens=20
)


--- Testing reproducibility with the SAME seed (42) ---

--- Demonstrating seed=42 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Clean with a Clear Conscience, Naturally."


--- Run 2 ---
Generated Text:


"Clean with a Clear Conscience, Naturally."


--- Run 3 ---
Generated Text:


"Clean with a Clear Conscience, Naturally."


--- End Demonstration for this parameter setting ---


In [32]:
# Example 7.2: Different seeds, same prompt, same parameters (to get diverse options for review)
print("\n--- Testing different seeds for DIVERSE slogan options ---")
generate_and_display(
    common_prompt_seed,
    "seed", 101,
    num_runs=1, # One run for each distinct seed is enough to show variety
    temperature=0.7,
    max_tokens=20
)


--- Testing different seeds for DIVERSE slogan options ---

--- Demonstrating seed=101 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Sparkle with a Clear Conscience, Clean with Green."


--- End Demonstration for this parameter setting ---


In [33]:
# Example 7.3: Different seeds, same prompt, same parameters (to get diverse options for review)
generate_and_display(
    common_prompt_seed,
    "seed", 202,
    num_runs=1,
    temperature=0.7,
    max_tokens=20
)


--- Demonstrating seed=202 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Clean with a Clear Conscience, Inside and Out."


--- End Demonstration for this parameter setting ---


In [34]:
# Example 7.4: Different seeds, same prompt, same parameters (to get diverse options for review)
generate_and_display(
    common_prompt_seed,
    "seed", 303,
    num_runs=1,
    temperature=0.7,
    max_tokens=20
)


--- Demonstrating seed=303 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Clean with a Clear Conscience, Naturally."


--- End Demonstration for this parameter setting ---
